In [1]:
import sys
sys.path.append("../")

##### import library

In [2]:
from typing import List, Optional
import requests
import pandas as pd
import mygene
import utility
import importlib
importlib.reload(utility)
from utility import resolve_ensembl_ids_with_fallback, get_chembl_ids_fast,  get_disease_ids_fast, validate_id_dataframe
import pickle

In [3]:
dct_result = dict()

##### Read pickle

In [4]:
with open("Llama_same_question_response_final.pkl", "rb") as f:
    Llama_same_question_response = pickle.load(f)


In [5]:
for model in ['llama-3.3-70b-versatile', 'gpt-5-nano', 'grok-4-1-fast-non-reasoning-latest', "gemini-2.5-flash-lite", "OpenTarget", "BioChatter", "ctd", "ttd", "hcdt"]:

    
    if model=="llama-3.3-70b-versatile":
        dct_result[model] = Llama_same_question_response[model]

    # if model=="gpt-5-nano":
    #     dct_result[model] = OpenAI_same_question_response[model]

    # if model=='grok-4-1-fast-non-reasoning-latest':
    #     dct_result[model] = Grok_same_question_response[model]

    # if model=="gemini-2.5-flash-lite":
    #     dct_result[model] = Gemini_same_question_response[model]

    # if model=="OpenTarget":
    #     dct_result[model] = Opentarget_same_question_response[model]

    # if model=="BioChatter":
    #     dct_result[model] = Biochatter_same_question_response[model]

    # if model=="ctd":
    #     dct_result[model] = CTD_same_question_response[model]

    # if model=="hcdt":
    #     dct_result[model] = HCDT_same_question_response[model]


##### Verify if only valid columns are present

In [6]:
allowed = {"gene_name", "drug_name", "disease_name", "pathway_name"}

missing_df = []
not_dataframe = []
empty_df = []
bad_columns = []

total_payloads = 0

for model, queries in dct_result.items():
    if not isinstance(queries, dict):
        continue

    for question, runs in queries.items():
        if not isinstance(runs, dict):
            continue

        for run_id, payload in runs.items():
            total_payloads += 1

            if not isinstance(payload, dict) or "dataframe" not in payload:
                missing_df.append({
                    "model": model,
                    "question": question,
                    "run_id": run_id,
                    "payload_type": type(payload).__name__
                })
                continue

            df = payload.get("dataframe")

            if not isinstance(df, pd.DataFrame):
                not_dataframe.append({
                    "model": model,
                    "question": question,
                    "run_id": run_id,
                    "df_type": type(df).__name__
                })
                continue

            if df.empty:
                empty_df.append({
                    "model": model,
                    "question": question,
                    "run_id": run_id,
                    "columns": list(df.columns)
                })

            cols = {str(c) for c in df.columns}
            extra = sorted(cols - allowed)

            if extra:
                bad_columns.append({
                    "model": model,
                    "question": question,
                    "run_id": run_id,
                    "columns": list(df.columns),
                    "extra_columns": extra
                })

print(f"Total payloads checked: {total_payloads}")
print(f"Missing dataframe key: {len(missing_df)}")
print(f"Dataframe is not pd.DataFrame: {len(not_dataframe)}")
print(f"Empty dataframes: {len(empty_df)}")
print(f"Dataframes with disallowed columns: {len(bad_columns)}")

bad_columns_df = pd.DataFrame(bad_columns)
missing_df_df = pd.DataFrame(missing_df)
not_dataframe_df = pd.DataFrame(not_dataframe)
empty_df_df = pd.DataFrame(empty_df)

display(bad_columns_df)
display(missing_df_df)
display(not_dataframe_df)
display(empty_df_df)


Total payloads checked: 350
Missing dataframe key: 0
Dataframe is not pd.DataFrame: 0
Empty dataframes: 0
Dataframes with disallowed columns: 0


""


""


""


""


In [7]:
dct_jaccard = dict()

question_entities = {
    "gene_name": set(),
    "drug_name": set(),
    "disease_name": set(),
}

allowed_cols = {"gene_name", "drug_name", "disease_name"}

for model, queries in dct_result.items():
    dct_jaccard[model] = {}
    print(f"\nWorking for model: {model}")

    for question, runs in queries.items():

        for run_id, payload in runs.items():
            df = payload.get("dataframe")

            # ---- validation ----
            if not isinstance(df, pd.DataFrame) or df.empty:
                continue

            # ---- skip pathway outputs ----
            if "pathway_name" in df.columns:
                continue

            # ---- column sanity ----
            if not set(df.columns).issubset(allowed_cols):
                continue

            # ---- collect entities globally ----
            for col in allowed_cols:
                if col in df.columns:
                    values = (
                        df[col]
                        .dropna()
                        .astype(str)
                        .str.strip()
                        .tolist()
                    )
                    question_entities[col].update(values)

# ---- finalize (sets → sorted lists) ----
for k in question_entities:
    question_entities[k] = sorted(question_entities[k])


Working for model: llama-3.3-70b-versatile


##### Associate id with all gene entry

In [8]:
# Resolve unique gene symbols to IDs (MyGene -> OpenTargets fallback).
df_gene = resolve_ensembl_ids_with_fallback(
    list(question_entities["gene_name"]),
    use_opentargets_fallback=True,
)
df_gene = df_gene.drop_duplicates(subset=["gene_name", "gene_id"]).reset_index(drop=True)
df_gene


2026-02-27 18:44:06,698 [WARNING] Input sequence provided is already in string format. No operation performed
2026-02-27 18:44:06,699 [WARNING] Input sequence provided is already in string format. No operation performed
2026-02-27 18:44:06,700 [INFO] querying 1-278 ...
2026-02-27 18:44:08,587 [INFO] HTTP Request: POST https://mygene.info/v3/query/ "HTTP/1.1 200 OK"
2026-02-27 18:44:09,590 [INFO] Finished.
2026-02-27 18:44:09,597 [WARNING] 2 input query terms found dup hits:	[('C9ORF72', 2), ('MTOR', 2)]
2026-02-27 18:44:09,598 [WARNING] 20 input query terms found no hit:	['ANDP32', 'BAT3', 'C21ORF2', 'CTL4', 'FLK1', 'GBA', 'IL28B', 'MS4A', 'NRAMP1', 'OSTALPHA', 'PAD14',


[warn] Could not resolve 'ANDP32' via MyGene or OpenTargets. Possible typo or ambiguous gene-family prefix.
[warn] Could not resolve 'MS4A' via MyGene or OpenTargets. Possible typo or ambiguous gene-family prefix.
[warn] Could not resolve 'PAD14' via MyGene or OpenTargets. Possible typo or ambiguous gene-family prefix.
[warn] Could not resolve 'PGFRA' via MyGene or OpenTargets. Possible typo or ambiguous gene-family prefix.


,gene_name,gene_id,source
0,ABCA7,ENSG00000064687,mygene
1,ABL1,ENSG00000097007,mygene
2,ABL2,ENSG00000143322,mygene
3,AKT1,ENSG00000142208,mygene
4,ALK,ENSG00000171094,mygene
...,...,...,...
273,VEGFR1,ENSG00000102755,opentargets
274,VEGFR2,ENSG00000128052,opentargets
275,VEGFR3,ENSG00000037280,opentargets
276,VHL,ENSG00000134086,mygene


In [9]:
df_gene[df_gene.isna().any(axis=1)]

,gene_name,gene_id,source
6,ANDP32,NaN,None
162,MS4A,NaN,None
188,PAD14,NaN,None
195,PGFRA,NaN,None


##### Associate id with all drug entry

In [10]:
# Resolve unique drug surface forms to IDs.
# This keeps original raw names as rows and writes mapped ID + source.
df_drug = await get_chembl_ids_fast(list(question_entities["drug_name"]))
df_drug = df_drug.drop_duplicates(subset=["drug_name", "drug_id"]).reset_index(drop=True)
df_drug


2026-02-27 18:44:30,518 [INFO] [LLM] Total 8 names split into 1 batches (max_batch_chars=6000, max_batch_items=80, model=llama-3.3-70b-versatile).
2026-02-27 18:44:30,520 [INFO] [LLM] Batch size: 8


[map][drug][OpenTargets] raw='AZathioprine' -> id='CHEMBL1542'
[map][drug][OpenTargets] raw='Abatacept' -> id='CHEMBL1201823'
[map][drug][OpenTargets] raw='Adalimumab' -> id='CHEMBL1201580'
[map][drug][OpenTargets] raw='Ado-trastuzumab emtansine' -> id='CHEMBL1743082'
[map][drug][OpenTargets] raw='Afatinib' -> id='CHEMBL1173655'
[map][drug][OpenTargets] raw='Alfacalcidol' -> id='CHEMBL1601669'
[map][drug][OpenTargets] raw='Almotriptan' -> id='CHEMBL1505'
[map][drug][OpenTargets] raw='Amantadine' -> id='CHEMBL660'
[map][drug][OpenTargets] raw='Apomorphine' -> id='CHEMBL53'
[map][drug][OpenTargets] raw='Aspirin' -> id='CHEMBL25'
[map][drug][OpenTargets] raw='Atezolizumab' -> id='CHEMBL3707227'
[map][drug][OpenTargets] raw='Axicabtagene ciloleucel' -> id='CHEMBL3989989'
[map][drug][OpenTargets] raw='Azathioprine' -> id='CHEMBL1542'
[map][drug][OpenTargets] raw='Belimumab' -> id='CHEMBL1789843'
[map][drug][OpenTargets] raw='Bevacizumab' -> id='CHEMBL1201583'
[map][drug][OpenTargets] raw='B

2026-02-27 18:44:31,012 [INFO] HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-02-27 18:44:31,035 [INFO] [LLM] Batch done in 0.52s
2026-02-27 18:44:31,036 [INFO] [OT] Verifying 1 unique drug suggestions...
2026-02-27 18:44:31,433 [INFO] [OT] 'paclitaxel' → 'PACLITAXEL' (CHEMBL428647)
2026-02-27 18:44:31,435 [INFO] Done: 1 / 8 confirmed | 7 → None


[map][drug][Llama] raw='Nab-paclitaxel' -> canonical='PACLITAXEL' -> id='CHEMBL428647'


,drug_name,drug_id,source
0,AZathioprine,CHEMBL1542,OpenTargets
1,Abatacept,CHEMBL1201823,OpenTargets
2,Adalimumab,CHEMBL1201580,OpenTargets
3,Ado-trastuzumab emtansine,CHEMBL1743082,OpenTargets
4,Afatinib,CHEMBL1173655,OpenTargets
...,...,...,...
137,Vinorelbine,CHEMBL553025,OpenTargets
138,Zolmitriptan,CHEMBL1185,OpenTargets
139,abatacept,CHEMBL1201823,OpenTargets
140,cabozantinib,CHEMBL2105717,OpenTargets


In [11]:
df_drug["source"].value_counts()

source
OpenTargets    134
Llama            1
Name: count, dtype: int64

In [12]:
# df_drug[df_drug["source"]=="Llama"]

In [13]:
df_drug[df_drug.isna().any(axis=1)]

,drug_name,drug_id,source
8,Aminosalicylates,None,None
32,Corticosteroids,None,None
56,Glucocorticoids,None,None
62,Immunomodulators,None,None
68,JAK inhibitors,None,None
89,Nonsteroidal Anti-Inflammatory Drugs,None,None
120,TNF-alpha inhibitors,None,None


##### Associate id with all disease entry

In [14]:
# Resolve unique disease surface forms to IDs.
# Canonical fallback is internal; returned rows remain keyed by raw input disease_name.
df_disease = await get_disease_ids_fast(list(question_entities["disease_name"]))
df_disease = df_disease.drop_duplicates(subset=["disease_name", "disease_id"]).reset_index(drop=True)
df_disease


2026-02-27 18:44:32,770 [INFO] [LLM] Total 15 names split into 1 batches (max_batch_chars=6000, max_batch_items=80, model=llama-3.3-70b-versatile).
2026-02-27 18:44:32,771 [INFO] [LLM] Batch size: 15


[map][disease][OpenTargets] raw='Acute Lymphoblastic Leukemia' -> id='EFO_0000220'
[map][disease][OpenTargets] raw='Acute Myeloid Leukemia' -> id='EFO_0000222'
[map][disease][OpenTargets] raw='Alzheimer's disease' -> id='MONDO_0007088'
[map][disease][OpenTargets] raw='Amyotrophic Lateral Sclerosis' -> id='MONDO_0004976'
[map][disease][OpenTargets] raw='Anaplastic Large Cell Lymphoma' -> id='EFO_0003032'
[map][disease][OpenTargets] raw='Ankylosing Spondylitis' -> id='EFO_0003898'
[map][disease][OpenTargets] raw='Apert Syndrome' -> id='MONDO_0007041'
[map][disease][OpenTargets] raw='Apert syndrome' -> id='MONDO_0007041'
[map][disease][OpenTargets] raw='Autism Spectrum Disorder' -> id='EFO_0003756'
[map][disease][OpenTargets] raw='Beare-Stevenson Syndrome' -> id='MONDO_0007412'
[map][disease][OpenTargets] raw='Bladder Cancer' -> id='MONDO_0004986'
[map][disease][OpenTargets] raw='Brain Cancer' -> id='MONDO_0001657'
[map][disease][OpenTargets] raw='Brain cancer' -> id='MONDO_0001657'
[map]

2026-02-27 18:44:33,374 [INFO] HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-02-27 18:44:33,377 [INFO] [LLM] Batch done in 0.60s
2026-02-27 18:44:33,378 [INFO] [LLM] 'Acute Pain' → None
2026-02-27 18:44:33,378 [INFO] [LLM] 'Acute pain' → None
2026-02-27 18:44:33,379 [INFO] [LLM] 'Advanced Renal Cell Carcinoma' → 'renal cell carcinoma'
2026-02-27 18:44:33,379 [INFO] [LLM] 'Deficient Mismatch Repair Cancer' → None
2026-02-27 18:44:33,380 [INFO] [LLM] 'Gaastric Cancer' → 'stomach cancer'
2026-02-27 18:44:33,380 [INFO] [LLM] 'Gastroesophageal Cancer' → 'gastroesophageal cancer'
2026-02-27 18:44:33,381 [INFO] [LLM] 'Microsatellite Instability-High Cancer' → None
2026-02-27 18:44:33,381 [INFO] [LLM] 'Mismatch Repair Deficient Colorectal Cancer' → None
2026-02-27 18:44:33,382 [INFO] [LLM] 'Parkinson’s disease' → "parkinson's disease"
2026-02-27 18:44:33,382 [INFO] [LLM] 'Primary Dysmenorrhea' → None
2026-02-27 18:44:33,383 [INFO] [LLM] 'Primary Medi

[map][disease][Llama] raw='Advanced Renal Cell Carcinoma' -> canonical='renal cell carcinoma' -> id='EFO_0000681'
[map][disease][Llama] raw='Gaastric Cancer' -> canonical='stomach neoplasm' -> id='EFO_0003897'
[map][disease][Llama] raw='Parkinson’s disease' -> canonical='Parkinson disease' -> id='MONDO_0005180'
[map][disease][Llama] raw='Triple Negative Breast Cancer' -> canonical='triple-negative breast cancer' -> id='EFO_0005537'
[warn] Unresolved diseases after OpenTargets/Llama/OLS: ['Acute Pain', 'Acute pain', 'Deficient Mismatch Repair Cancer', 'Gastroesophageal Cancer', 'Microsatellite Instability-High Cancer', 'Mismatch Repair Deficient Colorectal Cancer', 'Primary Dysmenorrhea', 'Primary Mediastinal Large B-Cell Lymphoma', 'Radioactive Iodine Refractory Thyroid Cancer', 'Tumor Mutational Burden-High Cancer', 'disease_name']


,disease_name,disease_id,source
0,Acute Lymphoblastic Leukemia,EFO_0000220,OpenTargets
1,Acute Myeloid Leukemia,EFO_0000222,OpenTargets
2,Acute Pain,None,None
3,Acute pain,None,None
4,Advanced Renal Cell Carcinoma,EFO_0000681,Llama
...,...,...,...
120,pancreatic adenocarcinoma,EFO_1000044,OpenTargets
121,renal cell carcinoma,EFO_0000681,OpenTargets
122,rheumatoid arthritis,EFO_0000685,OpenTargets
123,tuberculosis,MONDO_0018076,OpenTargets


In [15]:
df_disease["source"].value_counts()

source
OpenTargets    110
Llama            4
Name: count, dtype: int64

In [16]:
df_disease[df_disease.isna().any(axis=1)]

,disease_name,disease_id,source
2,Acute Pain,None,None
3,Acute pain,None,None
30,Deficient Mismatch Repair Cancer,None,None
42,Gastroesophageal Cancer,None,None
67,Microsatellite Instability-High Cancer,None,None
69,Mismatch Repair Deficient Colorectal Cancer,None,None
89,Primary Dysmenorrhea,None,None
90,Primary Mediastinal Large B-Cell Lymphoma,None,None
95,Radioactive Iodine Refractory Thyroid Cancer,None,None
110,Tumor Mutational Burden-High Cancer,None,None


In [17]:
# ------------------------------------------------------------------
# Disease canonical audit (OpenTargets-strict IDs only)
# ------------------------------------------------------------------
# IMPORTANT:
# - This cell runs an additional Llama canonicalization pass over unresolved rows.
# - IDs are still resolved ONLY via OpenTargets mapIds (score==1).
# - To avoid changing the baseline resolver output, writeback is OFF by default.
APPLY_AUDIT_MAPPINGS = False

# 1) Rows unresolved after primary resolver
unresolved_rows = df_disease[df_disease["disease_id"].isna()].reset_index(drop=True)
unresolved_raw_names = list(
    dict.fromkeys(
        str(name).strip()
        for name in unresolved_rows["disease_name"].dropna().astype(str).tolist()
        if str(name).strip()
    )
)

# 2) Llama canonicalization suggestions
disease_input_map = {name: None for name in unresolved_raw_names}
canonicalisation_map = await utility.canonicalise_disease_dict(disease_input_map)

# 3) OpenTargets strict verification (score==1 only)
canonical_terms_norm = list(
    dict.fromkeys(
        utility._normalize_disease_term(v)
        for v in canonicalisation_map.values()
        if isinstance(v, str) and v.strip()
    )
)
canonical_to_id = utility.resolve_diseases_opentargets_bulk(
    canonical_terms_norm,
    strict_score_one=True,
)

# 4) Build persistent audit table
audit_rows = []
applied_count = 0
candidate_count = 0

for raw_disease_name in unresolved_raw_names:
    canonical_name = canonicalisation_map.get(raw_disease_name)
    if isinstance(canonical_name, str) and canonical_name.strip():
        canonical_name = canonical_name.strip()
        canonical_norm = utility._normalize_disease_term(canonical_name)
        resolved_id = canonical_to_id.get(canonical_norm)
    else:
        canonical_name = None
        canonical_norm = None
        resolved_id = None

    can_apply = bool(resolved_id)
    if can_apply:
        candidate_count += 1
        print(f"[audit][disease][Llama->OT] raw='{raw_disease_name}' -> canonical='{canonical_name}' -> id='{resolved_id}'")

    if can_apply and APPLY_AUDIT_MAPPINGS:
        mask = df_disease["disease_name"].astype(str).str.strip() == raw_disease_name
        df_disease.loc[mask, "disease_id"] = resolved_id
        df_disease.loc[mask, "source"] = "Llama"
        applied_count += 1

    status = (
        "mappable_ot_exact" if can_apply else
        ("no_canonical" if canonical_name is None else "canonical_not_in_ot")
    )
    audit_rows.append({
        "raw_disease_name": raw_disease_name,
        "canonical_name": canonical_name,
        "canonical_norm": canonical_norm,
        "resolved_disease_id": resolved_id,
        "status": status,
        "can_apply": can_apply,
        "applied": bool(can_apply and APPLY_AUDIT_MAPPINGS),
    })

disease_canonical_audit_df = pd.DataFrame(audit_rows)
if not disease_canonical_audit_df.empty:
    disease_canonical_audit_df = disease_canonical_audit_df.sort_values(
        ["status", "raw_disease_name"],
        ascending=[True, True],
    ).reset_index(drop=True)

print(f"[summary][disease][audit] OT-mappable candidates: {candidate_count}")
print(f"[summary][disease][audit] applied to df_disease: {applied_count} (APPLY_AUDIT_MAPPINGS={APPLY_AUDIT_MAPPINGS})")
if not disease_canonical_audit_df.empty:
    print("[summary][disease][audit] status counts:")
    print(disease_canonical_audit_df["status"].value_counts(dropna=False))

disease_canonical_audit_df


2026-02-27 18:44:34,366 [INFO] [LLM] Total 11 names split into 1 batches (max_batch_chars=6000, max_batch_items=80, model=llama-3.3-70b-versatile).
2026-02-27 18:44:34,367 [INFO] [LLM] Batch size: 11
2026-02-27 18:44:35,007 [INFO] HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-02-27 18:44:35,010 [INFO] [LLM] Batch done in 0.64s
2026-02-27 18:44:35,011 [INFO] [LLM] 'Acute Pain' → None
2026-02-27 18:44:35,011 [INFO] [LLM] 'Acute pain' → None
2026-02-27 18:44:35,012 [INFO] [LLM] 'Deficient Mismatch Repair Cancer' → None
2026-02-27 18:44:35,012 [INFO] [LLM] 'Gastroesophageal Cancer' → 'gastroesophageal cancer'
2026-02-27 18:44:35,013 [INFO] [LLM] 'Microsatellite Instability-High Cancer' → None
2026-02-27 18:44:35,013 [INFO] [LLM] 'Mismatch Repair Deficient Colorectal Cancer' → None
2026-02-27 18:44:35,014 [INFO] [LLM] 'Primary Dysmenorrhea' → 'primary dysmenorrhea'
2026-02-27 18:44:35,014 [INFO] [LLM] 'Primary Mediastinal Large B-Cell Lymphoma' → 

[summary][disease][audit] OT-mappable candidates: 0
[summary][disease][audit] applied to df_disease: 0 (APPLY_AUDIT_MAPPINGS=False)
[summary][disease][audit] status counts:
status
no_canonical    11
Name: count, dtype: int64


,raw_disease_name,canonical_name,canonical_norm,resolved_disease_id,status,can_apply,applied
0,Acute Pain,None,None,None,no_canonical,False,False
1,Acute pain,None,None,None,no_canonical,False,False
2,Deficient Mismatch Repair Cancer,None,None,None,no_canonical,False,False
3,Gastroesophageal Cancer,None,None,None,no_canonical,False,False
4,Microsatellite Instability-High Cancer,None,None,None,no_canonical,False,False
5,Mismatch Repair Deficient Colorectal Cancer,None,None,None,no_canonical,False,False
6,Primary Dysmenorrhea,None,None,None,no_canonical,False,False
7,Primary Mediastinal Large B-Cell Lymphoma,None,None,None,no_canonical,False,False
8,Radioactive Iodine Refractory Thyroid Cancer,None,None,None,no_canonical,False,False
9,Tumor Mutational Burden-High Cancer,None,None,None,no_canonical,False,False


In [18]:
# disease_canonical_audit_df

In [19]:
df_disease[df_disease["source"]=='Llama']

,disease_name,disease_id,source
4,Advanced Renal Cell Carcinoma,EFO_0000681,Llama
39,Gaastric Cancer,EFO_0003897,Llama
84,Parkinson’s disease,MONDO_0005180,Llama
107,Triple Negative Breast Cancer,EFO_0005537,Llama


In [20]:
df_disease["source"].value_counts()

source
OpenTargets    110
Llama            4
Name: count, dtype: int64

In [21]:
df_disease

,disease_name,disease_id,source
0,Acute Lymphoblastic Leukemia,EFO_0000220,OpenTargets
1,Acute Myeloid Leukemia,EFO_0000222,OpenTargets
2,Acute Pain,None,None
3,Acute pain,None,None
4,Advanced Renal Cell Carcinoma,EFO_0000681,Llama
...,...,...,...
120,pancreatic adenocarcinoma,EFO_1000044,OpenTargets
121,renal cell carcinoma,EFO_0000681,OpenTargets
122,rheumatoid arthritis,EFO_0000685,OpenTargets
123,tuberculosis,MONDO_0018076,OpenTargets


In [22]:
# df_disease[df_disease["source"]=="Llama"]

In [23]:
def compute_jaccard_from_run_dataframes(dct_result, df_gene, df_disease, df_drug):
    """
    Compute per-question cross-run Jaccard consistency after mapping entities to IDs.

    For each (model, question):
    - intersection: entries present in ALL valid runs
    - union: entries present in ANY valid run
    - jaccard = |intersection| / |union|

    All comparisons are CASE-INDEPENDENT and ID-based.

    Side effects:
    - Stores per-run mapped dataframe in:
        dct_result[model][question][run]['dataframe_id']

    Returns:
    - dct_jaccard[model][question] = {
          'jaccard': float,
          'n_valid_runs': int,
          'intersection': set[tuple],
          'union': set[tuple],
      }
    """

    dct_jaccard = {}

    # ------------------------------------------------------------------
    # Build deterministic one-to-one mapping tables
    # ------------------------------------------------------------------
    gene_map = (
        df_gene.assign(_gene_norm=df_gene["gene_name"].astype(str).str.strip().str.upper())
        [["_gene_norm", "gene_id"]]
        .dropna(subset=["gene_id"])
        .drop_duplicates("_gene_norm", keep="first")
    )

    disease_map = (
        df_disease.assign(_disease_norm=df_disease["disease_name"].astype(str).str.strip().str.upper())
        [["_disease_norm", "disease_id"]]
        .dropna(subset=["disease_id"])
        .drop_duplicates("_disease_norm", keep="first")
    )

    drug_map = (
        df_drug.assign(_drug_norm=df_drug["drug_name"].astype(str).str.strip().str.upper())
        [["_drug_norm", "drug_id"]]
        .dropna(subset=["drug_id"])
        .drop_duplicates("_drug_norm", keep="first")
    )

    # ------------------------------------------------------------------
    # Main loop
    # ------------------------------------------------------------------
    for model, model_payload in dct_result.items():
        dct_jaccard[model] = {}
        print(f"\nWorking for model: {model}")

        for question_key, runs_dict in model_payload.items():
            run_sets = {}

            for run_number, run_payload in runs_dict.items():
                df = run_payload.get("dataframe")

                # ---- basic validity ----
                if not isinstance(df, pd.DataFrame) or df.empty:
                    continue

                if "pathway_name" in df.columns:
                    continue

                work_df = df.copy()

                # print(work_df.head())

                final_cols = []

                # ---- gene ----
                if "gene_name" in work_df.columns:
                    work_df["_gene_norm"] = (
                        work_df["gene_name"].astype(str).str.strip().str.upper()
                    )
                    work_df = work_df.merge(gene_map, on="_gene_norm", how="left")
                    final_cols.append("gene_id")

                # ---- disease ----
                if "disease_name" in work_df.columns:
                    work_df["_disease_norm"] = (
                        work_df["disease_name"].astype(str).str.strip().str.upper()
                    )
                    work_df = work_df.merge(disease_map, on="_disease_norm", how="left")
                    final_cols.append("disease_id")

                # ---- drug ----
                if "drug_name" in work_df.columns:
                    work_df["_drug_norm"] = (
                        work_df["drug_name"].astype(str).str.strip().str.upper()
                    )
                    work_df = work_df.merge(drug_map, on="_drug_norm", how="left")
                    final_cols.append("drug_id")

                if not final_cols:
                    continue

                # ---- canonical ID dataframe ----
                id_df = (
                    work_df[final_cols]
                    .dropna(how="any")
                    .drop_duplicates()
                    .reset_index(drop=True)
                )

                dct_result[model][question_key][run_number]["dataframe_id"] = id_df

                if id_df.empty:
                    continue

                # ---- CASE-INDEPENDENT tuple set ----
                run_set = {
                    tuple(str(v).strip().upper() for v in row)
                    for row in id_df.itertuples(index=False, name=None)
                }

                if run_set:
                    run_sets[run_number] = run_set

            # ------------------------------------------------------------------
            # Compute intersection / union
            # ------------------------------------------------------------------
            n_valid_runs = len(run_sets)

            if n_valid_runs < 2:
                print(
                    f"Not enough valid runs for '{question_key}' "
                    f"({n_valid_runs}/{len(runs_dict)}). Skipping."
                )
                continue

            sets = list(run_sets.values())
            intersection = set.intersection(*sets)
            union = set.union(*sets)

            if not union:
                print(f"Empty union for '{question_key}', skipping.")
                continue

            jaccard = len(intersection) / len(union)

            dct_jaccard[model][question_key] = {
                "jaccard": jaccard,
                "n_valid_runs": n_valid_runs,
                "intersection": intersection,
                "union": union,
            }

            print(
                f"'{question_key}': "
                f"J={jaccard:.4f}, "
                f"|∩|={len(intersection)}, "
                f"|∪|={len(union)}, "
                f"runs={n_valid_runs}"
            )

    return dct_jaccard


dct_jaccard = compute_jaccard_from_run_dataframes(
    dct_result=dct_result,
    df_gene=df_gene,
    df_disease=df_disease,
    df_drug=df_drug,
)


Working for model: llama-3.3-70b-versatile
'Which diseases are associated with BRAF?': J=0.4286, |∩|=3, |∪|=7, runs=5
'Which genes are associated with amyotrophic lateral sclerosis?': J=0.2500, |∩|=10, |∪|=40, runs=5
'Which diseases are associated with CDK4?': J=0.7500, |∩|=3, |∪|=4, runs=5
Not enough valid runs for 'Which pathways are associated with the JAK2 gene?' (0/5). Skipping.
'Which drugs are used to treat rheumatoid arthritis?': J=1.0000, |∩|=10, |∪|=10, runs=5
'Which genes are associated with ovarian cancer?': J=0.2143, |∩|=3, |∪|=14, runs=5
'What are the known targets of regorafenib?': J=0.2000, |∩|=3, |∪|=15, runs=5
Not enough valid runs for 'Which pathways are associated with the STAT3 gene?' (0/5). Skipping.
'Which genes are associated with fever (pyrexia)?': J=0.7500, |∩|=3, |∪|=4, runs=2
'Which diseases are treated with bevacizumab?': J=0.4444, |∩|=4, |∪|=9, runs=5
'What are the known targets of cabozantinib?': J=0.6250, |∩|=5, |∪|=8, runs=5
Not enough valid runs for '

In [24]:
# Run Jaccard consistency computation across runs after ID grounding.
dct_jaccard = compute_jaccard_from_run_dataframes(
    dct_result=dct_result,
    df_gene=df_gene,
    df_disease=df_disease,
    df_drug=df_drug,
)

# Flatten into a quick summary table for inspection.
rows = [
    {
        "model": model,
        "question": question,
        "jaccard": payload["jaccard"],
        "n_valid_runs": payload["n_valid_runs"],
        "intersection_size": len(payload["intersection"]),
        "union_size": len(payload["union"]),
    }
    for model, qmap in dct_jaccard.items()
    for question, payload in qmap.items()
]

jaccard_summary = pd.DataFrame(rows)
if jaccard_summary.empty:
    print("No valid Jaccard entries were produced. Check input runs and validation filters.")
else:
    display(jaccard_summary.sort_values(["model", "jaccard"], ascending=[True, False]).head(20))



Working for model: llama-3.3-70b-versatile
'Which diseases are associated with BRAF?': J=0.4286, |∩|=3, |∪|=7, runs=5
'Which genes are associated with amyotrophic lateral sclerosis?': J=0.2500, |∩|=10, |∪|=40, runs=5
'Which diseases are associated with CDK4?': J=0.7500, |∩|=3, |∪|=4, runs=5
Not enough valid runs for 'Which pathways are associated with the JAK2 gene?' (0/5). Skipping.
'Which drugs are used to treat rheumatoid arthritis?': J=1.0000, |∩|=10, |∪|=10, runs=5
'Which genes are associated with ovarian cancer?': J=0.2143, |∩|=3, |∪|=14, runs=5
'What are the known targets of regorafenib?': J=0.2000, |∩|=3, |∪|=15, runs=5
Not enough valid runs for 'Which pathways are associated with the STAT3 gene?' (0/5). Skipping.
'Which genes are associated with fever (pyrexia)?': J=0.7500, |∩|=3, |∪|=4, runs=2
'Which diseases are treated with bevacizumab?': J=0.4444, |∩|=4, |∪|=9, runs=5
'What are the known targets of cabozantinib?': J=0.6250, |∩|=5, |∪|=8, runs=5
Not enough valid runs for '

,model,question,jaccard,n_valid_runs,intersection_size,union_size
3,llama-3.3-70b-versatile,Which drugs are used to treat rheumatoid arthr...,1.000000,5,10,10
14,llama-3.3-70b-versatile,For which diseases is darolutamide used?,1.000000,5,1,1
16,llama-3.3-70b-versatile,Which diseases are associated with the KRAS gene?,1.000000,5,3,3
22,llama-3.3-70b-versatile,Which drugs are used to treat chronic myeloid ...,1.000000,5,5,5
25,llama-3.3-70b-versatile,Which targets are acted on by imatinib?,1.000000,5,4,4
27,llama-3.3-70b-versatile,Which biological targets does tofacitinib inte...,1.000000,5,4,4
31,llama-3.3-70b-versatile,Which diseases are associated with HTT?,1.000000,5,1,1
32,llama-3.3-70b-versatile,Which genes are associated with breast cancer?,1.000000,5,5,5
34,llama-3.3-70b-versatile,Which genes are associated with cystic fibrosis?,1.000000,5,1,1
43,llama-3.3-70b-versatile,What are the known targets of abatacept?,1.000000,5,2,2


In [25]:
# Persist enriched run payloads with dataframe_id attached for each run.
with open("Llama_same_question_response_with_id.pkl", "wb") as f:
    pickle.dump(dct_result, f)
